In [2]:
!pip install xgboost

In [20]:
import numpy as np
from collections import Counter
from sklearn.model_selection import train_test_split
# 如果xgboost装不上，用from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import f1_score

# 1. 假数据（或用你之前的docs）
docs = [
   "机器学习 是 人工智能 的 一个 分支",
"深度学习 是 机器学习 的 一个 子集",
"自然语言处理 是 人工智能 的 应用",
"计算机视觉 是 人工智能 的 重要 方向",
"大语言模型 是 深度学习 的 产物",
"神经网络 是 深度学习 的 基础 组件",
"卷积神经网络 是 计算机视觉 的 常用 模型",
"循环神经网络 擅长 处理 序列 类型 的 数据",
"Transformer 是 大模型 的 核心 架构",
"注意力机制 是 Transformer 的 关键 模块",
"监督学习 需要 使用 带标签 的 训练 数据",
"无监督学习 主要 挖掘 数据 的 内在 模式",
"强化学习 通过 环境反馈 不断 优化 策略",
"半监督学习 同时 使用 少量标签 和 大量无标签数据",
"过拟合 指 模型 在训练集表现好 泛化能力差",
"欠拟合 指 模型 无法 捕捉 数据 的 基本规律",
"正则化 用于 抑制 模型 的 过拟合 现象",
"交叉验证 用来 评估 模型 的 泛化性能",
"梯度下降 是 训练模型 的 主流 优化算法",
"损失函数 用来 衡量 预测值 和真实值 的差距",
"激活函数 给网络 引入 非线性 的表达能力",
"数据集 一般 划分 训练集 验证集 测试集",
"特征工程 是 传统机器学习 的 重要步骤",
"词袋模型 是 早期 的 文本 表示 方法",
"词嵌入 将词语 映射 到低维 的向量空间",
"RAG 技术 将外部知识库 和大模型 相结合",
"提示词工程 用来 引导大模型 输出 指定结果",
"生成式AI 可以 创建文本 图像 音频等内容",
"判别式模型 主要 完成分类 判别类任务",
"生成式模型 学习 数据 的整体分布规律",
"池化层 用来 降低卷积网络 的特征维度",
"归一化 可以 加速神经网络 的训练收敛",
"批量归一化 缓解深度网络 的梯度消失问题",
"梯度消失 深层网络 参数 难以更新 的现象",
"梯度爆炸 梯度数值过大 导致训练不稳定",
"预训练 让模型 先学习通用知识 再做微调",
"微调 在预训练模型基础上 适配下游任务",
"多模态模型 可以 同时处理文字 图片 音频",
"目标检测 在图像中 定位并且识别物体",
"语义分割 对图像每一个像素 做类别划分",
"知识图谱 结构化存储 现实世界实体与关系",
"推理阶段 是模型训练完成后 做预测的过程",
"训练阶段 模型 通过数据 迭代更新权重参数",
"嵌入向量 可以用来 计算文本之间 的相似度"
    # ... 多造几篇，至少20篇，分两类 ...
]

labels = [0,0,1,1,1,0,1,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1,0,0,0,0,0,0,0,0,0,1,1,1,1,0,0,0]
  # 0=机器学习类，1=人工智能类

# 2. TF-IDF（用你之前手写的代码，或直接用sklearn的TfidfVectorizer）
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer = TfidfVectorizer(tokenizer=lambda x: x.split(),token_pattern=None)
X = vectorizer.fit_transform(docs).toarray()
y = np.array(labels)

# 3. 划分train/test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42,stratify=y)

# 4. 模型1：你的逻辑回归（简化版）
f1_lr=[]
def sigmoid(z):
    return 1/(1+np.exp(-z))

def compute_loss(X_b,y,w):
    z=X_b@w
    y_prob=sigmoid(z)

    loss=-np.mean(y*np.log(y_prob+1e-8)+(1-y)*np.log(1-y_prob+1e-8))
    return loss
# ... 用你之前的逻辑回归代码训练 ...
##L2正则化，防止过拟合
lam=0.01
#####################

n_train = X_train.shape[0]
X_train_b = np.hstack([np.ones((n_train, 1)), X_train])

n_test = X_test.shape[0]
X_test_b = np.hstack([np.ones((n_test, 1)), X_test])

lr = 0.3
epochs = 1000
w = np.zeros((X_train_b.shape[1], ))
loss_history = []
m = X_train_b.shape[0]

for i in range(epochs):
    z = X_train_b @ w
    y_prob = sigmoid(z)
    loss = compute_loss(X_train_b, y_train, w)
    loss_history.append(loss)
    
    grad = (1/m) * X_train_b.T @ (y_prob - y_train)+lam*w
    w = w - lr * grad
    
    if i % 200 == 0:
        print(f"epoch:{i:4d}, loss:{loss:.4f}")
y_pred_lr = (sigmoid(X_test_b @ w) >= 0.25).astype(int)
f1_lr = f1_score(y_test, y_pred_lr)

# 5. 模型2：XGBoost（调包）
model = XGBClassifier()
model.fit(X_train, y_train)
y_pred_xgb = model.predict(X_test)
f1_xgb = f1_score(y_test, y_pred_xgb)

print(f"逻辑回归 F1: {f1_lr:.4f}")
print(f"XGBoost F1:  {f1_xgb:.4f}")
print("测试集真实标签 y_test =", y_test)
print("逻辑回归预测 y_pred_lr =", y_pred_lr)


epoch:   0, loss:0.6931
epoch: 200, loss:0.3962
epoch: 400, loss:0.3365
epoch: 600, loss:0.3161
epoch: 800, loss:0.3086
逻辑回归 F1: 0.5455
XGBoost F1:  0.0000
测试集真实标签 y_test = [0 0 1 0 1 0 0 1 0]
逻辑回归预测 y_pred_lr = [1 1 1 1 1 1 1 1 0]
